## Memory Qubits in Q#

This notebook illustrates an advanced feature in Q#: memory qubits. Memory qubits can store a quantum state, but gates or operations cannot be applied to them. We call regular qubits that do support these operations "compute qubits." This distinction makes it possible to design algorithms that use fewer compute qubits, because memory qubits can use more efficient error-correction codes (see, for example, [yoked surface codes](https://arxiv.org/pdf/2312.04522)).

By default, all qubits are allocated as compute qubits. Users can turn a compute qubit into a memory qubit by calling `Std.Memory.Store`; the qubit is then "stored in memory," and no operations can be applied to it. When an operation needs to be performed on a qubit, it must be "loaded" from memory by calling `Std.Memory.Load`.

To see the effect of memory qubits on resource estimation, users need to call `Std.ResourceEstimation.EnableMemoryComputeArchitecture(0, 2)`. Here, the second parameter is the "strategy," and value `2` means the "manual" strategy, where the user explicitly moves qubits between memory and compute qubits by calling `Load` and `Store`. The first argument is "capacity" and is irrelevant for this strategy.

### Example: GHZ state preparation and measurement

In the example below, we prepare the [GHZ state](https://en.wikipedia.org/wiki/Greenberger%E2%80%93Horne%E2%80%93Zeilinger_state) on 10 memory qubits. We allocate qubits one-by-one (as compute qubits) and move them to memory as soon as each has been used as the control qubit of a CNOT gate. As a result, this circuit needs only 2 compute qubits. It also uses 10 memory qubits (to store the state), so in total it uses 12 qubits.

Then we demonstrate how to measure a memory register by moving qubits from memory to compute one-by-one, measuring, resetting, and releasing them.

We have to allocate and release qubits using operations from `QIR.Runtime` instead of `use`, because if we write `use q = Qubit[10]`, all 10 qubits will be allocated as compute qubits. In that case, the resource estimator would report 10 compute qubits, losing the benefit of using memory qubits.

In [1]:
from qdk import qsharp

In [2]:
%%qsharp

// Prepares GHZ state (|0..0> + |1..1>)/sqrt(2) in a memory register of length n.
// Uses at most 2 compute qubits.
// Qubits returned by this operation must be freed manually.
operation PrepareGhzStateInMemory(n: Int) : Qubit[] {
    mutable q = [QIR.Runtime.__quantum__rt__qubit_allocate()];    
    H(q[0]);
    for i in 1..n-1 {
        set q += [QIR.Runtime.__quantum__rt__qubit_allocate()];    
        CNOT(q[i-1], q[i]);
        Std.Memory.Store(q[i-1]);
    }
    Std.Memory.Store(q[n-1]);
    return q;
}

// Measures the value in a memory-qubit register, interpreted as a little-endian integer.
// Uses a single compute qubit as a buffer.
// In the end, the memory-qubit register is reset and freed.
operation MeasureMemoryAndRelease(mem : Qubit[]) : BigInt {
    mutable base : BigInt = 1L;
    mutable ans : BigInt = 0L;
    for i in 0..Length(mem)-1 {
        Std.Memory.Load(mem[i]);
        if (MResetZ(mem[i]) == One) {
            set ans += base;
        }
        set base *= 2L;
        QIR.Runtime.__quantum__rt__qubit_release(mem[i]);
    }
    return ans;
}

operation Main() : BigInt {
    Std.ResourceEstimation.EnableMemoryComputeArchitecture(0, 2);  // 2 = "Manual".
    let q = PrepareGhzStateInMemory(10);
    return MeasureMemoryAndRelease(q);
}

operation MainNoMemoryCompute() : BigInt {
    let q = PrepareGhzStateInMemory(10);
    return MeasureMemoryAndRelease(q);
}

In [3]:
from collections import Counter
Counter(qsharp.run("Main()", shots=20))

Counter({1023: 11, 0: 9})

In [4]:
qsharp.logical_counts("Main()")

{'numQubits': 12,
 'tCount': 0,
 'rotationCount': 0,
 'rotationDepth': 0,
 'cczCount': 0,
 'ccixCount': 0,
 'measurementCount': 10,
 'numComputeQubits': 2,
 'readFromMemoryCount': 10,
 'writeToMemoryCount': 10}

`logical_counts` reports 2 compute qubits and 12 total qubits, which implies 10 memory qubits, as expected.

For comparison, if we do not call `EnableMemoryComputeArchitecture`, we get 10 total qubits without a breakdown into memory and compute, which means 10 compute qubits:

In [5]:
qsharp.logical_counts("MainNoMemoryCompute()")

{'numQubits': 10,
 'tCount': 0,
 'rotationCount': 0,
 'rotationDepth': 0,
 'cczCount': 0,
 'ccixCount': 0,
 'measurementCount': 10}